# 02 — Supervised Models: Random Forest & XGBoost

This notebook visualises the results from Phase 2 (`run_supervised.py`).  
It **loads pre-trained models and the evaluation JSON** — it does not retrain.

Plots produced (all saved to `reports/`):
1. Confusion Matrix — Random Forest
2. Confusion Matrix — XGBoost
3. Per-category F1 comparison bar chart (RF vs XGBoost)
4. Feature importance comparison — top 15 features (RF vs XGBoost)
5. ROC curves (one-vs-rest) — one subplot per model

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

PROJECT_ROOT = Path().resolve().parent   # notebooks/ → project root
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.random_forest import RandomForestDetector
from src.models.xgboost_model import XGBoostDetector

# ── Paths ─────────────────────────────────────────────────────────────────────
REPORTS   = PROJECT_ROOT / 'reports'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
MODELS    = PROJECT_ROOT / 'data' / 'models'

EVAL_JSON  = REPORTS / 'supervised_evaluation.json'
RF_IMP     = REPORTS / 'rf_feature_importances.csv'
XGB_IMP    = REPORTS / 'xgb_feature_importances.csv'
RF_MODEL   = MODELS  / 'rf_detector.joblib'
XGB_MODEL  = MODELS  / 'xgb_detector.joblib'

# ── Style ─────────────────────────────────────────────────────────────────────
plt.style.use('dark_background')
PALETTE   = {'RF': '#4FC3F7', 'XGB': '#FF8A65'}
FIG_DPI   = 150

print('✅ Setup complete')

In [ ]:
# ── Load evaluation JSON ──────────────────────────────────────────────────────
with open(EVAL_JSON) as f:
    eval_data = json.load(f)

rf_eval  = eval_data['models']['random_forest']
xgb_eval = eval_data['models']['xgboost']
meta     = eval_data['metadata']
classes  = rf_eval['confusion_matrix_labels']   # ordered list of class strings

print(f"Classes ({len(classes)}): {classes}")
print(f"RF  macro F1 : {rf_eval['macro_f1']}   FPR: {rf_eval['fpr']}")
print(f"XGB macro F1 : {xgb_eval['macro_f1']}   FPR: {xgb_eval['fpr']}")

## Plot 1 & 2 — Confusion Matrices

In [ ]:
def plot_confusion_matrix(cm_list, labels, title, save_path, cmap='Blues'):
    cm = np.array(cm_list)
    # Normalise rows → recall per class
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.patch.set_facecolor('#1a1a2e')

    for ax, data, subtitle in zip(
        axes,
        [cm, cm_norm],
        ['Counts', 'Row-normalised (Recall per class)']
    ):
        ax.set_facecolor('#16213e')
        sns.heatmap(
            data, annot=True,
            fmt='.2f' if 'norm' in subtitle.lower() else 'd',
            xticklabels=labels, yticklabels=labels,
            cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='#333355',
            cbar_kws={'shrink': 0.75}
        )
        ax.set_title(subtitle, color='white', fontsize=11, pad=10)
        ax.set_xlabel('Predicted', color='#aaaacc', fontsize=10)
        ax.set_ylabel('True', color='#aaaacc', fontsize=10)
        ax.tick_params(colors='#ccccdd', labelrotation=40, labelsize=8)

    fig.suptitle(title, color='white', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved → {save_path}')


plot_confusion_matrix(
    rf_eval['confusion_matrix'], classes,
    'Random Forest — Confusion Matrix',
    REPORTS / 'rf_confusion_matrix.png'
)

plot_confusion_matrix(
    xgb_eval['confusion_matrix'], classes,
    'XGBoost — Confusion Matrix',
    REPORTS / 'xgb_confusion_matrix.png'
)

## Plot 3 — Per-category F1 Comparison (RF vs XGBoost)

In [ ]:
all_cats = sorted(set(rf_eval['per_class'].keys()) | set(xgb_eval['per_class'].keys()))

rf_f1s   = [rf_eval['per_class'].get(c, {}).get('f1', 0.0)  for c in all_cats]
xgb_f1s  = [xgb_eval['per_class'].get(c, {}).get('f1', 0.0) for c in all_cats]

x = np.arange(len(all_cats))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

bars_rf  = ax.bar(x - width/2, rf_f1s,  width, label='Random Forest', color=PALETTE['RF'],  alpha=0.9, edgecolor='white', linewidth=0.5)
bars_xgb = ax.bar(x + width/2, xgb_f1s, width, label='XGBoost',       color=PALETTE['XGB'], alpha=0.9, edgecolor='white', linewidth=0.5)

# Annotate bars
for bar in bars_rf:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
            ha='center', va='bottom', fontsize=7, color='#aaaacc')
for bar in bars_xgb:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
            ha='center', va='bottom', fontsize=7, color='#aaaacc')

ax.set_xticks(x)
ax.set_xticklabels(all_cats, rotation=35, ha='right', color='#ccccdd', fontsize=9)
ax.set_ylabel('F1 Score', color='#ccccdd', fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title('Per-category F1: Random Forest vs XGBoost', color='white', fontsize=13, fontweight='bold', pad=12)
ax.legend(facecolor='#22223b', edgecolor='#555577', labelcolor='white', fontsize=10)
ax.tick_params(axis='y', colors='#ccccdd')
ax.spines[:].set_color('#333355')
ax.yaxis.grid(True, linestyle='--', alpha=0.3, color='#555577')
ax.set_axisbelow(True)

# Add macro F1 line references
ax.axhline(rf_eval['macro_f1'],  color=PALETTE['RF'],  linestyle='--', linewidth=1.2, alpha=0.7, label=f"RF macro F1={rf_eval['macro_f1']}")
ax.axhline(xgb_eval['macro_f1'], color=PALETTE['XGB'], linestyle='--', linewidth=1.2, alpha=0.7, label=f"XGB macro F1={xgb_eval['macro_f1']}")
ax.legend(facecolor='#22223b', edgecolor='#555577', labelcolor='white', fontsize=9)

plt.tight_layout()
save_path = REPORTS / 'f1_comparison.png'
plt.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Saved → {save_path}')

## Plot 4 — Feature Importance Comparison (Top 15)

In [ ]:
rf_imp  = pd.read_csv(RF_IMP).head(15)
xgb_imp = pd.read_csv(XGB_IMP).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('Feature Importances — Top 15 Features\n(RF: impurity-based Gini  |  XGBoost: gain-based)',
             color='white', fontsize=12, fontweight='bold', y=1.01)

for ax, df, label, color in [
    (axes[0], rf_imp,  'Random Forest\n(Mean Decrease in Gini Impurity)', PALETTE['RF']),
    (axes[1], xgb_imp, 'XGBoost\n(Average Gain per Split)',              PALETTE['XGB']),
]:
    ax.set_facecolor('#16213e')
    bars = ax.barh(df['feature'][::-1], df['importance'][::-1], color=color, alpha=0.85, edgecolor='white', linewidth=0.4)
    ax.set_title(label, color='white', fontsize=10, pad=8)
    ax.set_xlabel('Importance', color='#aaaacc', fontsize=9)
    ax.tick_params(colors='#ccccdd', labelsize=8)
    ax.spines[:].set_color('#333355')
    ax.xaxis.grid(True, linestyle='--', alpha=0.3, color='#555577')
    ax.set_axisbelow(True)

    # Add value labels
    for bar in bars:
        w = bar.get_width()
        ax.text(w * 1.01, bar.get_y() + bar.get_height()/2, f'{w:.4f}',
                va='center', fontsize=7, color='#aaaacc')

plt.tight_layout()
save_path = REPORTS / 'feature_importance_comparison.png'
plt.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Saved → {save_path}')
print()
print('Note: RF uses impurity (Gini) — inflated for high-cardinality features (sport, dsport).')
print('      XGB uses gain — reflects average loss reduction when the feature is actually used.')
print('      Disagreements between the two are expected and informative.')

## Plot 5 — ROC Curves (One-vs-Rest per attack category)

In [ ]:
# Load test data to compute per-class ROC curves from raw probabilities
# (we need predict_proba output, not just the stored metrics)
feature_names = meta['feature_names']

print('Loading test.parquet for ROC curve computation…')
test_df = pd.read_parquet(PROCESSED / 'test.parquet', columns=feature_names + ['attack_cat'])
X_test  = test_df[feature_names].values.astype(np.float32)
y_test  = test_df['attack_cat'].values
print(f'  {X_test.shape[0]:,} test rows loaded')

print('Loading trained models…')
rf  = RandomForestDetector.load_model(RF_MODEL)
xgb = XGBoostDetector.load_model(XGB_MODEL)
print('  Models loaded ✅')

In [ ]:
def plot_roc_curves(model, X_test, y_test, classes, title, save_path, palette_color):
    """Plot one-vs-rest ROC curve per class in a grid of subplots."""
    proba = model._model.predict_proba(X_test)

    # Handle XGBoost label ordering (classes are stored in _classes not model.classes_)
    if hasattr(model, '_classes') and model._classes is not None:
        ordered_classes = model._classes
    else:
        ordered_classes = model._model.classes_

    y_bin = label_binarize(y_test, classes=ordered_classes)
    n_cls = len(ordered_classes)

    ncols = 5
    nrows = int(np.ceil(n_cls / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3.2))
    fig.patch.set_facecolor('#1a1a2e')
    axes = axes.flatten()

    for i, cls in enumerate(ordered_classes):
        ax = axes[i]
        ax.set_facecolor('#16213e')
        if y_bin[:, i].sum() == 0:
            ax.text(0.5, 0.5, 'No positives', ha='center', va='center', color='#888')
            ax.set_title(cls, color='white', fontsize=9)
            continue

        fpr_c, tpr_c, _ = roc_curve(y_bin[:, i], proba[:, i])
        roc_auc_c = auc(fpr_c, tpr_c)

        ax.plot(fpr_c, tpr_c, color=palette_color, linewidth=1.8, label=f'AUC={roc_auc_c:.3f}')
        ax.plot([0, 1], [0, 1], linestyle='--', color='#555577', linewidth=1)
        ax.fill_between(fpr_c, tpr_c, alpha=0.15, color=palette_color)

        ax.set_title(cls, color='white', fontsize=9, fontweight='bold')
        ax.legend(fontsize=8, facecolor='#22223b', labelcolor='white', edgecolor='#555577')
        ax.set_xlabel('FPR', color='#aaaacc', fontsize=8)
        ax.set_ylabel('TPR', color='#aaaacc', fontsize=8)
        ax.tick_params(colors='#ccccdd', labelsize=7)
        ax.spines[:].set_color('#333355')
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1.02])

    # Hide unused subplots
    for j in range(n_cls, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(title, color='white', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved → {save_path}')


print('Plotting RF ROC curves (computing predict_proba on 508K rows — ~1-2 min)…')
plot_roc_curves(
    rf, X_test, y_test, classes,
    'Random Forest — One-vs-Rest ROC Curves',
    REPORTS / 'rf_roc_curves.png',
    PALETTE['RF']
)

print('Plotting XGBoost ROC curves…')
plot_roc_curves(
    xgb, X_test, y_test, classes,
    'XGBoost — One-vs-Rest ROC Curves',
    REPORTS / 'xgb_roc_curves.png',
    PALETTE['XGB']
)

## Summary

| Metric | Random Forest | XGBoost |
|---|---|---|
| Macro F1 | see `rf_eval['macro_f1']` | see `xgb_eval['macro_f1']` |
| Weighted F1 | see `rf_eval['weighted_f1']` | see `xgb_eval['weighted_f1']` |
| ROC-AUC (OvR) | see `rf_eval['roc_auc']` | see `xgb_eval['roc_auc']` |
| FPR | see `rf_eval['fpr']` | see `xgb_eval['fpr']` |

**Next step**: Phase 3 — Unsupervised models (Isolation Forest + LSTM Autoencoder).